# Notebook 2 – Prétraitement & Alignement
**Projet Computer Vision IG.2405 – 2026**

Ce notebook détaille le pipeline de prétraitement des images de formulaires :
1. **Binarisation** (seuillage Otsu)
2. **Deskew** : correction d'inclinaison via Transformée de Hough
3. **Normalisation** : recadrage sur la zone active + redimensionnement
4. **Correction de perspective** : pour les photos prises en biais

> Ces opérations utilisent uniquement des méthodes bas niveau (filtrage, Hough, morphologie).

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Imports du projet
from utils.form_aligner import (
    to_gray, binarize, find_active_area, deskew,
    warp_to_form, normalize_form_image, FORM_W, FORM_H
)
from utils.grid_decoder import normalize_page

FORM_DIR = os.path.join('PROJECT 2026 -DATABASE-20260518', 'FORM1')

# Choisir une image avec photo
img_files = sorted([f for f in os.listdir(FORM_DIR)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
SAMPLE_IMG = os.path.join(FORM_DIR, img_files[0])
print('Image choisie :', img_files[0])

## 1. Image brute

La photo prise par l'étudiant peut être :
- Inclinée (jusqu'à ±15°)
- De résolution variable
- Avec des bords parasites (table, main, ...)

In [ ]:
img_raw = cv2.imread(SAMPLE_IMG)
print(f'Dimensions brutes : {img_raw.shape[1]} × {img_raw.shape[0]} px')

plt.figure(figsize=(6, 9))
plt.imshow(cv2.cvtColor(img_raw, cv2.COLOR_BGR2RGB))
plt.title('Image brute', fontsize=12)
plt.axis('off')
plt.tight_layout()
plt.show()

## 2. Binarisation (Seuillage Otsu)

La méthode d'Otsu détermine automatiquement le seuil optimal en minimisant la variance intra-classe :

$$\sigma_W^2(t) = w_0(t)\,\sigma_0^2(t) + w_1(t)\,\sigma_1^2(t)$$

Un flou gaussien préalable réduit le bruit haute fréquence.

In [ ]:
gray = to_gray(img_raw)
binary = binarize(gray, blur_ksize=5)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(gray, cmap='gray')
axes[0].set_title('Niveaux de gris')
axes[0].axis('off')

axes[1].hist(gray.ravel(), bins=256, range=(0, 256), color='gray', alpha=0.7)
# Calculer le seuil Otsu
blurred = cv2.GaussianBlur(gray, (5, 5), 0)
thresh_val, _ = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
axes[1].axvline(thresh_val, color='red', linewidth=2, label=f'Seuil Otsu = {thresh_val:.0f}')
axes[1].set_title('Histogramme + seuil Otsu')
axes[1].legend()

axes[2].imshow(binary, cmap='gray')
axes[2].set_title('Image binarisée (encre=blanc)')
axes[2].axis('off')

plt.suptitle('Étape 1 : Binarisation Otsu', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Détection de l'inclinaison – Transformée de Hough

La **transformée de Hough** détecte les droites en accumulant les votes dans l'espace $(\rho, \theta)$ :

$$\rho = x\cos\theta + y\sin\theta$$

Les lignes horizontales dominantes du formulaire permettent d'estimer l'angle d'inclinaison.

In [ ]:
# Détection des contours avant Hough
edges = cv2.Canny(binary, 50, 150)

# Transformée de Hough
lines = cv2.HoughLines(edges, 1, np.pi / 180, threshold=80)

# Calculer l'angle dominant
angles = []
if lines is not None:
    for rho, theta in lines[:, 0]:
        angle = np.degrees(theta) - 90
        if abs(angle) < 45:
            angles.append(angle)

dominant_angle = float(np.median(angles)) if angles else 0.0
print(f'Angle dominant détecté : {dominant_angle:.2f}°')

# Visualiser les lignes détectées
vis_lines = cv2.cvtColor(binary, cv2.COLOR_GRAY2BGR)
if lines is not None:
    for rho, theta in lines[:30, 0]:
        a, b = np.cos(theta), np.sin(theta)
        x0, y0 = a * rho, b * rho
        pt1 = (int(x0 + 2000 * (-b)), int(y0 + 2000 * a))
        pt2 = (int(x0 - 2000 * (-b)), int(y0 - 2000 * a))
        cv2.line(vis_lines, pt1, pt2, (0, 0, 255), 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(edges, cmap='gray')
axes[0].set_title('Contours Canny')
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(vis_lines, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'Lignes Hough détectées (angle={dominant_angle:.1f}°)')
axes[1].axis('off')

plt.suptitle('Étape 2 : Transformée de Hough', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

if angles:
    plt.figure(figsize=(8, 3))
    plt.hist(angles, bins=30, color='steelblue', edgecolor='black')
    plt.axvline(dominant_angle, color='red', linewidth=2, label=f'Médiane={dominant_angle:.1f}°')
    plt.title('Distribution des angles détectés')
    plt.xlabel('Angle (degrés)')
    plt.legend()
    plt.tight_layout()
    plt.show()

## 4. Correction d'inclinaison (Deskew)

La correction s'effectue par une **rotation d'image** d'angle $-\hat{\theta}$ autour du centre :

$$M_{rot} = \begin{pmatrix} \cos\hat{\theta} & -\sin\hat{\theta} \\ \sin\hat{\theta} & \cos\hat{\theta} \end{pmatrix}$$

L'interpolation bilinéaire (`INTER_LINEAR`) préserve la qualité de l'image.

In [ ]:
img_deskewed = deskew(img_raw)

fig, axes = plt.subplots(1, 2, figsize=(12, 8))
axes[0].imshow(cv2.cvtColor(img_raw, cv2.COLOR_BGR2RGB))
axes[0].set_title('Avant deskew')
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(img_deskewed, cv2.COLOR_BGR2RGB))
axes[1].set_title('Après deskew')
axes[1].axis('off')

plt.suptitle('Étape 3 : Correction d\'inclinaison', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Détection de la zone active

La zone active est délimitée par les **marques de coin** du formulaire (brackets `L`).

Méthode :
1. Binarisation
2. **Composantes connexes** → filtrage par aire et rapport d'aspect
3. Boîte englobante des 4 coins candidats

In [ ]:
from utils.form_aligner import _find_corner_candidates

gray_d = to_gray(img_deskewed)
bin_d  = binarize(gray_d)
candidates = _find_corner_candidates(bin_d)

print(f'{len(candidates)} candidats de coin détectés')

# Visualiser les candidats
vis_corners = cv2.cvtColor(bin_d, cv2.COLOR_GRAY2BGR)
for (x, y, w, h) in candidates:
    cv2.rectangle(vis_corners, (x, y), (x + w, y + h), (0, 255, 0), 3)

# Zone active détectée
ax0, ay0, aw, ah = find_active_area(img_deskewed)
cv2.rectangle(vis_corners, (ax0, ay0), (ax0 + aw, ay0 + ah), (0, 0, 255), 4)

plt.figure(figsize=(7, 10))
plt.imshow(cv2.cvtColor(vis_corners, cv2.COLOR_BGR2RGB))
plt.title('Candidats coins (vert) + zone active (rouge)')
plt.axis('off')
plt.tight_layout()
plt.show()

## 6. Normalisation vers le repère du formulaire

Le formulaire est recadré et redimensionné vers une taille de référence **900 × 1270 px**.

Cette normalisation garantit que les coordonnées des ROIs (zones d'intérêt) sont
identiques pour toutes les images, indépendamment de la résolution ou de l'appareil.

In [ ]:
# Pipeline complet : deskew → détection zone active → resize vers 900×1270
img_normalized = normalize_page(img_deskewed)

print(f'Taille normalisée : {img_normalized.shape[1]} × {img_normalized.shape[0]} px')

fig, axes = plt.subplots(1, 3, figsize=(15, 7))

axes[0].imshow(cv2.cvtColor(img_raw, cv2.COLOR_BGR2RGB))
axes[0].set_title(f'Original ({img_raw.shape[1]}×{img_raw.shape[0]})')
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(img_deskewed, cv2.COLOR_BGR2RGB))
axes[1].set_title('Après deskew')
axes[1].axis('off')

axes[2].imshow(cv2.cvtColor(img_normalized, cv2.COLOR_BGR2RGB))
axes[2].set_title(f'Normalisé ({img_normalized.shape[1]}×{img_normalized.shape[0]})')
axes[2].axis('off')

plt.suptitle('Pipeline de normalisation complet', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Comparaison photo vs PDF

Après normalisation, l'image photo et l'image PDF doivent être dans le même repère.

In [ ]:
import fitz

pdf_files = sorted([f for f in os.listdir(FORM_DIR) if f.lower().endswith('.pdf')])
if pdf_files:
    sample_pdf = os.path.join(FORM_DIR, pdf_files[0])
    doc = fitz.open(sample_pdf)
    mat = fitz.Matrix(150/72, 150/72)
    pix = doc[0].get_pixmap(matrix=mat)
    pdf_img = np.frombuffer(pix.samples, np.uint8).reshape(pix.height, pix.width, pix.n)
    if pix.n == 4:
        pdf_img = cv2.cvtColor(pdf_img, cv2.COLOR_RGBA2BGR)
    doc.close()

    pdf_norm = normalize_page(pdf_img)

    fig, axes = plt.subplots(1, 2, figsize=(12, 8))
    axes[0].imshow(cv2.cvtColor(img_normalized, cv2.COLOR_BGR2RGB))
    axes[0].set_title('Photo normalisée')
    axes[0].axis('off')

    axes[1].imshow(cv2.cvtColor(pdf_norm, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f'PDF normalisé ({pdf_files[0]})')
    axes[1].axis('off')

    plt.suptitle('Résultat après normalisation : photo vs PDF', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

## Bilan

| Étape | Méthode | Module |
|---|---|---|
| Binarisation | Otsu + flou gaussien | `form_aligner.binarize` |
| Deskew | Hough → angle médian → rotation | `form_aligner.deskew` |
| Zone active | Composantes connexes (brackets L) | `form_aligner.find_active_area` |
| Normalisation | Recadrage + redimensionnement 900×1270 | `grid_decoder.normalize_page` |

**Prochaine étape** → `03_decodage_grille.ipynb` : lecture de la grille Student ID et des cases à cocher.